In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

import optuna

In [2]:
data_path = '../src/features.csv'

In [3]:
df = pd.read_csv(data_path, index_col=0)
df.head()

,record_id,r_peak,annotation_sample,type,dist_from_annotation,anomaly,mean,std,min,max,...,median,iqr,energy,rms,skew,kurtosis,max_abs,mean_abs,waveform_length,max_diff
0,100,370,370.0,N,0.0,0,-0.308917,0.234895,-0.535,0.940,...,-0.385,0.11500,27.108925,0.388079,3.807791,15.165259,0.940,0.370861,4.485,0.325
1,100,663,662.0,N,1.0,0,-0.331750,0.215952,-0.570,0.960,...,-0.395,0.08750,28.204775,0.395845,4.287391,19.807686,0.960,0.381583,4.600,0.470
2,100,947,946.0,N,1.0,0,-0.336667,0.204356,-0.645,0.860,...,-0.385,0.05000,27.919050,0.393835,4.160135,19.139860,0.860,0.381944,4.490,0.410
3,100,1231,1231.0,N,0.0,0,-0.338556,0.192977,-0.565,0.820,...,-0.395,0.08625,27.334800,0.389692,4.257258,19.927498,0.820,0.379056,4.390,0.390
4,100,1515,1515.0,N,0.0,0,-0.327778,0.204850,-0.545,0.885,...,-0.395,0.10500,26.892300,0.386525,4.114318,18.648272,0.885,0.373111,4.465,0.390


In [4]:
id_columns = [
    'record_id',
    'r_peak',
    'annotation_sample',
    'type',
    'dist_from_annotation',
    'anomaly'
]

feature_columns = [
    col for col in df.columns
    if col not in id_columns
]

In [5]:
print(feature_columns)

['mean', 'std', 'min', 'max', 'peak_to_peak', 'median', 'iqr', 'energy', 'rms', 'skew', 'kurtosis', 'max_abs', 'mean_abs', 'waveform_length', 'max_diff']


In [6]:
record_ids = df['record_id'].unique()

train_ids, temp_ids = train_test_split(
    record_ids,
    test_size=0.4,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=42
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 27
Validation: 9
Test: 10


In [7]:
train_df_all = df[
    df['record_id'].isin(train_ids)
].copy()

train_df = train_df_all[
    train_df_all['anomaly'] == 0
].copy()

val_df = df[
    df['record_id'].isin(val_ids)
].copy()

test_df = df[
    df['record_id'].isin(test_ids)
].copy()

In [8]:
train_df['anomaly'].value_counts()

anomaly
0    44564
Name: count, dtype: int64

In [9]:
val_df['anomaly'].value_counts()

anomaly
0    13753
1     2916
Name: count, dtype: int64

In [10]:
test_df['anomaly'].value_counts()

anomaly
0    14991
1     7040
Name: count, dtype: int64

In [11]:
X_train = train_df[feature_columns]

X_val = val_df[feature_columns]
y_val = val_df['anomaly']

X_test = test_df[feature_columns]
y_test = test_df['anomaly']

In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

In [13]:
def objective(trial):
    nu = trial.suggest_float('nu', 0.001, 0.2, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    kernel = trial.suggest_categorical('kernel', ['rbf', 'poly', 'sigmoid'])
    tol = trial.suggest_float('tol', 1e-5, 1e-2, log=True)
    
    params = {
        'kernel': kernel,
        'nu': nu,
        'gamma': gamma,
        'tol': tol
    }
    
    if kernel == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 5)
        params['coef0'] = trial.suggest_float('coef0', -1, 1)
    
    elif kernel == 'sigmoid':
        params['coef0'] = trial.suggest_float('coef0', -1, 1)
        
    model = OneClassSVM(**params)
    model.fit(X_train_scaled)
    
    pred = model.predict(X_val_scaled)
    y_pred = np.where(
        pred == -1,
        1,
        0
    )
    
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    trial.set_user_attr('precision', precision)
    trial.set_user_attr('recall', recall)
    
    return f1

In [14]:
study = optuna.create_study(
    direction='maximize'
)

[I 2026-09-09 10:43:05,136] A new study created in memory with name: no-name-7bfa4138-9550-4709-8ed6-2791b6e5d60d


In [15]:
study.optimize(
    objective,
    n_trials=30
)

[I 2026-09-09 10:43:10,200] Trial 0 finished with value: 0.15906053083826618 and parameters: {'nu': 0.06650094395565727, 'gamma': 0.0001022110962959976, 'kernel': 'sigmoid', 'tol': 0.006048972794024662, 'coef0': -0.3033249519247978}. Best is trial 0 with value: 0.15906053083826618.
[I 2026-09-09 10:43:10,360] Trial 1 finished with value: 0.0013364517206815904 and parameters: {'nu': 0.0014349013718849917, 'gamma': 0.02785462082831095, 'kernel': 'poly', 'tol': 0.0005641587015801433, 'degree': 5, 'coef0': 0.6341555446244258}. Best is trial 0 with value: 0.15906053083826618.
[I 2026-09-09 10:43:16,380] Trial 2 finished with value: 0.14886440296829323 and parameters: {'nu': 0.0563462736352657, 'gamma': 0.0010093081859046953, 'kernel': 'rbf', 'tol': 0.00011093491798514515}. Best is trial 0 with value: 0.15906053083826618.
[I 2026-09-09 10:43:23,724] Trial 3 finished with value: 0.3939845886154611 and parameters: {'nu': 0.0787219227342283, 'gamma': 0.0016351613961469252, 'kernel': 'poly', 'to

In [24]:
best_trial = study.best_trial

print("F1:", best_trial.value)
print("Precision:", best_trial.user_attrs['precision'])
print("Recall:", best_trial.user_attrs['recall'])
print("Params:", best_trial.params)

F1: 0.3939845886154611
Precision: 0.30896686159844056
Recall: 0.5435528120713305
Params: {'nu': 0.0787219227342283, 'gamma': 0.0016351613961469252, 'kernel': 'poly', 'tol': 0.0006770577690569524, 'degree': 3, 'coef0': 0.14697898816709043}


In [25]:
best_model = OneClassSVM(**best_trial.params)
best_model.fit(X_train_scaled)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.0016351613961469252
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.14697898816709043
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.0006770577690569524
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.0787219227342283
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1
Name,Type,Value


In [26]:
val_pred = best_model.predict(X_val_scaled)
y_val_pred = np.where(val_pred == -1, 1, 0)

cm = confusion_matrix(y_val, y_val_pred)

print(cm)

[[10208  3545]
 [ 1331  1585]]


In [27]:
val_results = val_df.copy()

val_results['prediction'] = y_val_pred

In [28]:
pd.crosstab(
    val_results['type'],
    val_results['prediction'],
    normalize='index'
)

prediction,0,1
type,,
A,0.769231,0.230769
E,1.000000,0.000000
F,0.500000,0.500000
L,0.273585,0.726415
N,0.742238,0.257762
R,1.000000,0.000000
V,0.831669,0.168331


In [29]:
type_results = val_results.groupby('type').agg(
    count=('type', 'size'),
    detected=('prediction', 'sum')
)

type_results['detection_rate'] = (
    type_results['detected']
    / type_results['count']
)

type_results

,count,detected,detection_rate
type,,,
A,13,3,0.230769
E,101,0,0.000000
F,2,1,0.500000
L,2014,1463,0.726415
N,13753,3545,0.257762
R,85,0,0.000000
V,701,118,0.168331


In [30]:
normal_val = val_results[
    val_results['type'] == 'N'
]

false_positive_rate = (
    normal_val['prediction'].mean()
)

print(false_positive_rate)

0.25776194284883297


In [31]:
pred = ocsvm.predict(X_test_scaled)
y_pred = np.where(pred == -1, 1, 0)

NameError: name 'ocsvm' is not defined

In [32]:
confusion_matrix(y_test, y_pred)

NameError: name 'y_pred' is not defined

In [ ]:
recall = recall_score(y_test, y_pred)

print("Recall:", recall)

Recall: 0.19606211869107043
